<a href="https://colab.research.google.com/github/Demian354678/exploratory-data-analysis-652784/blob/main/examen1_demian_652784.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Ejecutar si no se cuenta con estas librerías, en Google Colab no tendrán problemas
!pip install pandas_datareader statsmodels plotly --quiet

# Importar librerías
import pandas as pd
import numpy as np
from pandas_datareader import data as web
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing, Holt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_squared_error
from math import sqrt

# Traer datos
serie = web.DataReader('MRTSSM44811USS', 'fred', start='2010-01-01', end='2024-06-01')
serie = serie.rename(columns={'MRTSSM44811USS':'ventas'})
serie.dropna(inplace=True)

In [9]:
serie.shape

(134, 1)

In [11]:
fig = px.line(
    serie,
    y='ventas',
    title='Ventas Mensuales de NorthPeak'
)

fig.update_layout(
    xaxis_title='Fecha',
    yaxis_title='Ventas'
)

fig.show()

sales_nov_2017 = serie.loc['2017-11-01', 'ventas']
print(f"Ventas en Noviembre 2017: {int(sales_nov_2017)}")

Ventas en Noviembre 2017: 721


In [17]:
result = seasonal_decompose(
    serie['ventas'],
    model='additive',
    period=12
)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=['Serie original', 'Tendencia', 'Estacionalidad', 'Residuo'])

fig.add_trace(go.Scatter(x=result.observed.index, y=result.observed, name='Original'), row=1, col=1)
fig.add_trace(go.Scatter(x=result.trend.index, y=result.trend, name='Tendencia'), row=2, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal, name='Estacionalidad'), row=3, col=1)
fig.add_trace(go.Scatter(x=result.resid.index, y=result.resid, name='Residuo'), row=4, col=1)

fig.update_layout(height=900, title_text='Descomposición de la serie de tiempo')
fig.show()

residual_april_2020 = result.resid.loc['2020-04-01']
print(f"El valor del residual en Abril 2020 es: {residual_april_2020:.2f}")

El valor del residual en Abril 2020 es: -376.43


In [21]:
# Importar librerías principales
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing, Holt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy.stats import linregress

def graficar_serie(
        original,
        Forecast,
        title
):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=original.index,
            y=original,
            mode='lines',
            name='Pasajeros reales',
            opacity=0.5
        )
    )

    fig.add_trace(
        go.Scatter(
            x=Forecast.index,
            y=Forecast,
            mode='lines',
            name=title
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title='Fecha',
        yaxis_title='Pasajeros'
    )

    fig.show()

def mape(
    original,
    forecast
):

    mask = forecast.notna()

    mape = np.mean(
        np.abs(
            (original[mask] - forecast[mask]) / original[mask]
        )
    )*100

    return mape

In [22]:
hw_model = ExponentialSmoothing(
    serie['ventas'],
    trend='add',
    seasonal='add',
    seasonal_periods=12
).fit()

serie['HW'] = hw_model.fittedvalues

graficar_serie(
    serie['ventas'],
    serie['HW'],
    title='Metodo de Holt-Winters'
)

original_sales_april_2020 = serie.loc['2020-04-01', 'ventas']
forecast_sales_april_2020 = serie.loc['2020-04-01', 'HW']

difference_april_2020 = abs(original_sales_april_2020 - forecast_sales_april_2020)

print(f"La diferencia absoluta en Abril 2020 es: {difference_april_2020:.2f}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



La diferencia absoluta en Abril 2020 es: 225.02


In [24]:
mape_hw = mape(serie['ventas'], serie['HW'])
print(f"El MAPE del método de Holt-Winters es: {mape_hw:.2f}%")

El MAPE del método de Holt-Winters es: 6.66%


In [26]:
# TODO: Pronóstico futuro (6 meses)

n_forecast = 6

forecast = hw_model.forecast(n_forecast)

forecast

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index, y=serie['ventas'], mode='lines', name='Ventas reales'))
fig.add_trace(go.Scatter(x=serie.index, y=serie['HW'], mode='lines', name='Holt-Winters (ajuste)'))

future_dates = pd.date_range(serie.index[-1] + pd.DateOffset(months=1), periods=n_forecast, freq='MS')

fig.add_trace(go.Scatter(x=future_dates, y=forecast, mode='lines+markers', name='Pronóstico futuro', line=dict(dash='dash')))

fig.update_layout(title='Pronóstico de ventas con Holt-Winters', xaxis_title='Fecha', yaxis_title='Ventas')

fig.show()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.



In [27]:
forecast

,0
134,576.105694
135,550.129345
136,557.849051
137,575.201189
138,585.834388
139,583.291535
